# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Lane: Lane 4 — CTR / Engagement Opportunity Scoring** (carried over from Weeks 1–2). Unit of analysis and the `ctr_gap` proxy target are the same definitions used there; this notebook writes them down as a contract and checks every claim with a query against the real warehouse.

## 0. Setup — connect to the warehouse

Same pattern as `notebooks/03_working_with_the_full_release.ipynb`: DuckDB reads Parquet straight off Hugging Face, no download. Everything below iterates on **`month=2026-03`** — a mid-panel month. `fact_content_daily_performance_sample` is June 2026, the sealed final month; it's touched once at the very end only to prove the query mechanics work, never for the contract or the label logic.

In [3]:
%pip -q install duckdb

import os, getpass, duckdb

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",  # June 2026 -- SEALED, mechanics only
}

# Mid-panel month for every query in this notebook. Never the _sample table, and never a query
# that scans the full 78.8M-row fact table -- point straight at one partition instead.
MONTH = '2026-03'
MONTH_FACT = f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')"

# Cheap sanity checks -- dim_clients/dim_content are tiny, and DESCRIBE touches metadata only.
for name in ['dim_clients', 'dim_content']:
    n = con.sql(f"SELECT COUNT(*) FROM {TABLES[name]}").fetchone()[0]
    print(f"{name:14} {n:>10,} rows")

print("\ndim_content columns:")
print(con.sql(f"DESCRIBE SELECT * FROM {TABLES['dim_content']} LIMIT 0").df()['column_name'].tolist())
print("\nfact_content_daily_performance columns (month=2026-03):")
print(con.sql(f"DESCRIBE SELECT * FROM {MONTH_FACT} LIMIT 0").df()['column_name'].tolist())

# If a column name below doesn't match what just printed, this is the one place to fix it.
COLS = dict(client_id='client_hash_id', content_id='content_hash_id', date='report_date',
            impressions='gsc_impressions', clicks='gsc_clicks', position='gsc_avg_position',
            ga4_flag='ga4_data_available', content_type='content_type', main_intent='main_intent')


Paste your Hugging Face READ token (hf_...): ··········
dim_clients           104 rows
dim_content       519,606 rows

dim_content columns:
['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']

fact_content_daily_performance columns (month=2026-03):
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'session

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Raw table grain:** one row in `fact_content_daily_performance` = one `report_date` × one `client_hash_id` × one `content_hash_id` — a single day's search performance for a single content item at a single client. Verified in Query 1 below.

**My lane's unit of analysis** sits one level up, matching Weeks 1–2: one **(client_hash_id, content_hash_id) pair**, aggregated over a full month — a per-page-per-month CTR-gap score, not a per-page-per-day one.

**Time window: `month=2026-03`** — a mid-panel month. Per the warehouse warning, `fact_content_daily_performance_sample` is exactly June 2026, the final month and the natural outcome window of any past→future label — it's used only to sanity-check query mechanics (Section 3), never to build the contract or the label.

In [4]:
# Preview: pull real rows at the grain I'm claiming, before building anything on top of it.
preview = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id,
           gsc_impressions, gsc_clicks, gsc_avg_position, ga4_data_available
    FROM {MONTH_FACT}
    LIMIT 5
""").df()
preview


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_data_available
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,<NA>


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Tables:** `fact_content_daily_performance` (`month=2026-03` partition only) joined to `dim_content` on `content_hash_id` for static content metadata.

**What I'd predict / rank:** `ctr_gap = ctr − expected_ctr(position_tier)` — same definition as Week 2. `ctr` is this month's `clicks / impressions`; `expected_ctr(position_tier)` is the **median `ctr` within that same position tier, that same month**. It's a **defined-rule proxy**, not a future outcome: a persistent gap is a necessary condition for an opportunity, not proof one exists.

**One thing I deliberately exclude:** GA4 engagement columns (sessions, engaged sessions, pageviews aren't in this month's fact slice as clean numbers — only `ga4_data_available` is). Query 3 below shows that flag is real and worth respecting: before a client's `ga4_data_start`, GA4 columns are zero-filled, not genuinely zero. Folding GA4 in this week means either dropping a systematic slice of clients or silently reading "not tracked yet" as "no engagement." This lane starts CTR/position-only; GA4 engagement is a clean week-4+ extension once that coverage gap gets its own treatment.

**Full four-bucket classification:**

In [5]:
#%%
import pandas as pd

field_contract = pd.DataFrame([
    ("gsc_impressions (monthly SUM)",  "feature",          "observed search impressions, already occurred by month-end"),
    ("gsc_avg_position (monthly AVG)", "feature",          "observed search position, already occurred by month-end"),
    ("position_tier",                  "feature",          "deterministic bucket of avg_position"),
    ("content_type",                   "feature",          "static content metadata (dim_content), fixed at publish"),
    ("main_intent",                    "feature",          "static content metadata (dim_content), fixed at publish"),
    ("ctr (monthly clicks/impressions)","label ingredient","ctr_gap is computed directly from this -- never a model feature"),
    ("ctr_gap",                        "label / proxy",    "the target: ctr - expected_ctr(position_tier)"),
    ("client_hash_id",                 "context",          "join / group / split key only, never a feature"),
    ("content_hash_id",                "context",          "join / group / split key only, never a feature"),
    ("keyword_hash_id, url_hash_id",   "context",          "dim_content grouping/dedup keys only, per the lane guide"),
    ("ga4_* / ga4_data_available",     "excluded (this week)", "systematic pre-ga4_data_start coverage gap -- see note above"),
], columns=["field", "bucket", "why"])
field_contract


,field,bucket,why
0,gsc_impressions (monthly SUM),feature,"observed search impressions, already occurred ..."
1,gsc_avg_position (monthly AVG),feature,"observed search position, already occurred by ..."
2,position_tier,feature,deterministic bucket of avg_position
3,content_type,feature,"static content metadata (dim_content), fixed a..."
4,main_intent,feature,"static content metadata (dim_content), fixed a..."
5,ctr (monthly clicks/impressions),label ingredient,ctr_gap is computed directly from this -- neve...
6,ctr_gap,label / proxy,the target: ctr - expected_ctr(position_tier)
7,client_hash_id,context,"join / group / split key only, never a feature"
8,content_hash_id,context,"join / group / split key only, never a feature"
9,"keyword_hash_id, url_hash_id",context,"dim_content grouping/dedup keys only, per the ..."


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three small queries, all against the `month=2026-03` partition only (never the full 78.8M-row scan), then the five-feature frame, then the leakage trap.

### Query 1 — grain: one row really is one report_date × client × content

In [6]:
#%%
grain_probe = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {MONTH_FACT}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print(f"Grain violations in {MONTH}: {len(grain_probe)} (0 means the stated grain holds)")
grain_probe


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain violations in 2026-03: 0 (0 means the stated grain holds)


,report_date,client_hash_id,content_hash_id,c


### Query 2 — row count and date span for this slice

In [7]:
#%%
raw_counts = con.sql(f"""
    SELECT COUNT(*)                        AS n_rows,
           MIN(report_date)                AS first_day,
           MAX(report_date)                AS last_day,
           COUNT(DISTINCT client_hash_id)   AS n_clients,
           COUNT(DISTINCT content_hash_id)  AS n_content_items
    FROM {MONTH_FACT}
""").df()
print(f"Raw daily rows in {MONTH}:")
display(raw_counts)

# The lane's own slice: aggregated to (client, content)-per-month, 'visible' pages only
# (real position data + enough monthly volume to trust CTR).
lane_slice = con.sql(f"""
    WITH monthly AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS impressions,
               AVG(NULLIF(gsc_avg_position, 0)) AS avg_position
        FROM {MONTH_FACT}
        GROUP BY 1, 2
    )
    SELECT COUNT(*) AS n_pages, COUNT(DISTINCT client_hash_id) AS n_clients
    FROM monthly
    WHERE avg_position > 0 AND impressions >= 100
""").df()
print(f"\nVisible lane slice in {MONTH} (avg_position>0, monthly impressions>=100):")
lane_slice


Raw daily rows in 2026-03:


,n_rows,first_day,last_day,n_clients,n_content_items
0,9841378,2026-03-01,2026-03-31,55,331437


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Visible lane slice in 2026-03 (avg_position>0, monthly impressions>=100):


,n_pages,n_clients
0,101441,44


### Query 3 — availability, filtered with IS TRUE

In [8]:
#%%
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
        ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_available
    FROM {MONTH_FACT}
""").df()
print(f"GA4 availability in {MONTH} -- filtered with IS TRUE, not just truthy:")
availability


GA4 availability in 2026-03 -- filtered with IS TRUE, not just truthy:


,total_rows,ga4_available_rows,pct_available
0,9841378,413966.0,4.2


**Reading this:** whatever fraction survives `ga4_data_available IS TRUE` is the real, tracked-and-live share for this month — the rest isn't "zero engagement," it's "not tracked yet" for that client. This is exactly why GA4 is excluded from this week's feature set (Section 2) rather than quietly zero-filled.

### Five features (max) — the feature frame for this lane, this month

In [9]:
#%%
FEATURES = [
    ("monthly_impressions", "SUM(gsc_impressions) for the month",
     "knowable at the decision moment -- it only sums search impressions that already happened by month-end, no future data."),
    ("avg_position", "AVG(gsc_avg_position) over days with real position data",
     "knowable -- GSC position for days already reported is observed, not predicted."),
    ("position_tier", "bucket of avg_position (top_3 / page_1 / page_2_3 / deep)",
     "knowable -- a deterministic function of avg_position, which is itself already known."),
    ("content_type", "from dim_content",
     "knowable -- fixed content metadata set when the page was created, long before this scoring month."),
    ("main_intent", "from dim_content",
     "knowable -- static metadata for the same reason as content_type, not an outcome of the month being scored."),
]
for name, defn, why in FEATURES:
    print(f"- {name}: {defn}\n    -> {why}\n")


- monthly_impressions: SUM(gsc_impressions) for the month
    -> knowable at the decision moment -- it only sums search impressions that already happened by month-end, no future data.

- avg_position: AVG(gsc_avg_position) over days with real position data
    -> knowable -- GSC position for days already reported is observed, not predicted.

- position_tier: bucket of avg_position (top_3 / page_1 / page_2_3 / deep)
    -> knowable -- a deterministic function of avg_position, which is itself already known.

- content_type: from dim_content
    -> knowable -- fixed content metadata set when the page was created, long before this scoring month.

- main_intent: from dim_content
    -> knowable -- static metadata for the same reason as content_type, not an outcome of the month being scored.



In [10]:
#%%
feature_frame = con.sql(f"""
    WITH monthly AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS monthly_impressions,
               SUM(gsc_clicks)      AS monthly_clicks,
               AVG(NULLIF(gsc_avg_position, 0)) AS avg_position
        FROM {MONTH_FACT}
        GROUP BY 1, 2
    )
    SELECT m.client_hash_id, m.content_hash_id,
           m.monthly_impressions, m.avg_position,
           CASE WHEN m.avg_position <= 3  THEN 'top_3'
                WHEN m.avg_position <= 10 THEN 'page_1'
                WHEN m.avg_position <= 20 THEN 'page_2_3'
                ELSE 'deep' END AS position_tier,
           d.content_type, d.main_intent,
           m.monthly_clicks * 1.0 / NULLIF(m.monthly_impressions, 0) * 100 AS ctr
    FROM monthly m
    JOIN {TABLES['dim_content']} d USING (content_hash_id)
    WHERE m.avg_position > 0 AND m.monthly_impressions >= 100
""").df()

feature_frame['expected_ctr'] = feature_frame.groupby('position_tier')['ctr'].transform('median')
feature_frame['ctr_gap'] = feature_frame['ctr'] - feature_frame['expected_ctr']

print(f"Feature frame: {len(feature_frame):,} pages, {feature_frame['client_hash_id'].nunique()} clients")
feature_frame[['position_tier', 'content_type', 'main_intent', 'monthly_impressions',
               'avg_position', 'ctr', 'ctr_gap']].head(8)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame: 101,441 pages, 44 clients


,position_tier,content_type,main_intent,monthly_impressions,avg_position,ctr,ctr_gap
0,page_1,keyword article,informational,602.0,4.428747,0.664452,0.468996
1,page_1,keyword article,informational,810.0,4.866123,0.123457,-0.071999
2,top_3,keyword article,informational,1858.0,1.854929,0.322928,0.070403
3,page_1,keyword article,informational,536.0,4.442543,0.186567,-0.008888
4,page_1,keyword article,informational,496.0,4.018509,0.604839,0.409383
5,page_1,keyword article,informational,10849.0,8.240351,0.202784,0.007328
6,page_1,keyword article,informational,705.0,6.155424,0.141844,-0.053612
7,page_1,keyword article,informational,245.0,6.964106,0.000000,-0.195456


### The trap — add one label-derived column, watch the quick score jump, then delete it

`ctr_gap` is *computed from* `ctr` (`ctr_gap = ctr − expected_ctr(position_tier)`). Fold `ctr` itself into the model as a "feature" and watch what happens — this is notebook 02's leakage lesson, replayed here on real warehouse data.

In [11]:
#%%
from sklearn.linear_model import LinearRegression

honest_X = pd.get_dummies(
    feature_frame[['monthly_impressions', 'avg_position', 'position_tier', 'content_type', 'main_intent']],
    drop_first=True,
)
y = feature_frame['ctr_gap']

honest_model = LinearRegression().fit(honest_X, y)
honest_r2 = honest_model.score(honest_X, y)
print(f"Honest quick score (R^2), five features only: {honest_r2:.3f}")

# THE TRAP -- add ctr itself, the exact field ctr_gap is computed from.
leaky_X = honest_X.copy()
leaky_X['ctr'] = feature_frame['ctr']
leaky_model = LinearRegression().fit(leaky_X, y)
leaky_r2 = leaky_model.score(leaky_X, y)
print(f"Leaky quick score (R^2), with ctr added:   {leaky_r2:.3f}")
print(f"\nJump: {honest_r2:.3f} -> {leaky_r2:.3f}, from adding one label-derived column.")


Honest quick score (R^2), five features only: 0.010
Leaky quick score (R^2), with ctr added:   1.000

Jump: 0.010 -> 1.000, from adding one label-derived column.


**What happened:** once `ctr` sits in the model next to `position_tier`, the regression just re-derives the label's own arithmetic (`ctr_gap = ctr − a per-tier constant`) — R² jumps toward 1.0 not because the model learned anything about search behavior, but because it was handed the answer. That's the trap.

**Deleting it and keeping the honest number:**

In [12]:
#%%
del leaky_X, leaky_model, leaky_r2  # the leak doesn't survive past this cell
print(f"Final feature-only quick score (R^2), kept: {honest_r2:.3f}")

Final feature-only quick score (R^2), kept: 0.010


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation: the unbalanced panel.** History depth differs wildly per client (`dim_clients.gsc_data_start`), and this contract only looks at one cross-sectional month. `position_tier` and `expected_ctr(position_tier)` are recomputed fresh from whichever clients happen to be live in `month=2026-03` — so a page's `ctr_gap` here can't say whether that gap is new or has persisted for months, only that it exists *in this one month's snapshot*. Telling "genuinely under-capturing" apart from "still stabilizing early in tracking" needs more than one month of history, which is out of scope for this notebook and belongs to the signal-audit and validation weeks ahead.

In [13]:
#%%
panel_check = con.sql(f"""
    SELECT c.gsc_data_start,
           COUNT(DISTINCT f.content_hash_id) AS content_items_in_month
    FROM {MONTH_FACT} f
    JOIN {TABLES['dim_clients']} c USING (client_hash_id)
    GROUP BY 1
    ORDER BY 1
""").df()
print(f"Clients live in {MONTH}: {len(panel_check)}, "
      f"gsc_data_start ranges {panel_check['gsc_data_start'].min()} -> {panel_check['gsc_data_start'].max()}")
panel_check


Clients live in 2026-03: 36, gsc_data_start ranges 2025-01-27 00:00:00 -> 2026-03-27 00:00:00


,gsc_data_start,content_items_in_month
0,2025-01-27,7598
1,2025-02-11,29333
2,2025-03-11,11223
3,2025-06-07,25356
4,2025-06-18,4325
5,2025-06-21,15645
6,2025-06-29,10690
7,2025-07-01,31887
8,2025-07-06,1220
9,2025-07-07,1715


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.